# 03 — Conformance Checking

Process discovery (Inductive Miner IMf) + hand-crafted normative Petri net + token replay + alignments.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('figs', exist_ok=True)


## 2. Load & Convert to pm4py EventLog

In [ ]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv('data/'+
    'event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape after cleaning: {df.shape}')


Shape after cleaning: (1133314, 17)


In [3]:
import pm4py
from pm4py.objects.conversion.log import converter as log_converter

# Rename to pm4py standard columns
df_pm = df.rename(columns={
    'Case_id': 'case:concept:name',
    'Step': 'concept:name',
    'timestamp': 'time:timestamp',
    'Completed By': 'org:resource'
})
df_pm['hired_flag'] = df_pm['hired'].fillna(0).astype(int)
print('Columns:', df_pm.columns.tolist())


Columns: ['Job_Requisition_Code', 'Candidate_id', 'Recruiting Agency', 'Region', 'Country', 'Job Family', 'Job Family Group', 'case:concept:name', 'CW', 'Evergreen', 'Rejected', 'hired', 'org:resource', 'concept:name', 'Disposition Reason', 'All Stages for Candidate Current and Completed', 'time:timestamp', 'hired_flag']


## 3. Sampling (stratified by hired)

In [4]:
import numpy as np
np.random.seed(42)

def stratified_case_sample(df_pm, n, strat_col='hired_flag', random_state=42):
    from sklearn.model_selection import train_test_split
    case_labels = df_pm.drop_duplicates('case:concept:name')[['case:concept:name', strat_col]]
    if len(case_labels) <= n:
        return case_labels['case:concept:name'].tolist()
    _, sampled = train_test_split(
        case_labels,
        test_size=min(n, len(case_labels)-1),
        stratify=case_labels[strat_col],
        random_state=random_state
    )
    return sampled['case:concept:name'].tolist()

cases_disc  = stratified_case_sample(df_pm, 20_000)
cases_conf  = stratified_case_sample(df_pm, 50_000)
cases_align = stratified_case_sample(df_pm, 2_000)
print(f'Discovery : {len(cases_disc):,} cases')
print(f'Conformance: {len(cases_conf):,} cases')
print(f'Alignment : {len(cases_align):,} cases')


Discovery : 20,000 cases
Conformance: 50,000 cases
Alignment : 2,000 cases


In [5]:
def df_to_log(df_pm, case_ids):
    subset = df_pm[df_pm['case:concept:name'].isin(case_ids)].copy()
    # Drop rows where activity is NaN (kept by na=False filter) to avoid
    # mixed float/str in pm4py's internal get_alphabet() sorted() call
    subset = subset.dropna(subset=['concept:name'])
    subset['concept:name'] = subset['concept:name'].astype(str)
    subset['time:timestamp'] = pd.to_datetime(subset['time:timestamp'], utc=True)
    log = log_converter.apply(
        subset,
        parameters={log_converter.Variants.TO_EVENT_LOG.value.Parameters.CASE_ID_KEY: 'case:concept:name'},
        variant=log_converter.Variants.TO_EVENT_LOG
    )
    return log

log_discovery   = df_to_log(df_pm, cases_disc)
log_conformance = df_to_log(df_pm, cases_conf)
log_alignment   = df_to_log(df_pm, cases_align)
print('Logs created.')


Logs created.


## 4. Directly-Follows Graph (DFG) — top 50 edges

In [6]:
from pm4py.visualization.dfg import visualizer as dfg_vis

dfg, sa, ea = pm4py.discover_dfg(log_discovery)

# Filter to numeric counts only (guards against mixed-type DFG entries in some pm4py versions)
dfg_numeric = {k: v for k, v in dfg.items() if isinstance(v, (int, float))}
top50_edges = dict(sorted(dfg_numeric.items(), key=lambda x: x[1], reverse=True)[:50])

try:
    gviz = dfg_vis.apply(top50_edges, log=log_discovery,
                          parameters={dfg_vis.Variants.FREQUENCY.value.Parameters.FORMAT: 'png'})
    dfg_vis.save(gviz, 'figs/dfg_top50.png')
    print('DFG saved to figs/dfg_top50.png')
except Exception as e:
    print(f'DFG viz skipped (graphviz?): {e}')

print(f'DFG: {len(dfg):,} total edges, using top 50')


DFG saved to figs/dfg_top50.png
DFG: 746 total edges, using top 50


## 5. Process Discovery Algorithms

Five algorithms are applied to the 20 K-case discovery sample and compared:

| S | Algorithm | Key idea |
|---|-----------|----------|
| 5a | **Inductive Miner IMf** (noise=0.2) | Recursively splits the log on cut patterns; guarantees sound models |
| 5b | **Alpha Miner** | Builds a Petri net from directly-follows and causality relations; no noise handling |
| 5c | **Alpha+ Miner** | Extends Alpha to handle length-one/two loops |
| 5d | **Heuristics Miner** (dep=0.5) | Frequency-based dependency graph; robust to noise |
| 5e | **ILP Miner** | Integer-linear-programming causal footprint; exact but slow |
| 5f | **Process Tree** | Inductive process tree converted to Petri net (sound by construction) |

In [7]:
# -- 5a. Inductive Miner IMf (noise_threshold=0.2) -------------------------
# pm4py 2.7.x API: top-level function replaces the old variant-based call
net_disc, im_disc, fm_disc = pm4py.discover_petri_net_inductive(
    log_discovery, noise_threshold=0.2
)

print('Inductive Miner IMf (noise=0.2):')
print(f'  Places     : {len(net_disc.places)}')
print(f'  Transitions: {len(net_disc.transitions)}')
print(f'  Arcs       : {len(net_disc.arcs)}')

try:
    from pm4py.visualization.petri_net import visualizer as pn_vis
    gviz = pn_vis.apply(net_disc, im_disc, fm_disc, parameters={'format': 'png'})
    pn_vis.save(gviz, 'figs/petri_net_discovered_imf.png')
    print('Saved -> figs/petri_net_discovered_imf.png')
except Exception as e:
    print(f'Viz skipped: {e}')


Inductive Miner IMf (noise=0.2):
  Places     : 86
  Transitions: 146
  Arcs       : 322
Saved -> figs/petri_net_discovered_imf.png


### 5b. Alpha Miner

Classic algorithm by van der Aalst (2002). Derives causality from the directly-follows relation. Does **not** handle noise or duplicate labels -- expect a more complex net on real logs.

In [8]:
# -- 5b. Alpha Miner --------------------------------------------------------
try:
    net_alpha, im_alpha, fm_alpha = pm4py.discover_petri_net_alpha(log_discovery)
    print('Alpha Miner:')
    print(f'  Places     : {len(net_alpha.places)}')
    print(f'  Transitions: {len(net_alpha.transitions)}')
    print(f'  Arcs       : {len(net_alpha.arcs)}')
    try:
        from pm4py.visualization.petri_net import visualizer as pn_vis
        gviz = pn_vis.apply(net_alpha, im_alpha, fm_alpha, parameters={'format': 'png'})
        pn_vis.save(gviz, 'figs/petri_net_alpha.png')
        print('Saved -> figs/petri_net_alpha.png')
    except Exception as e:
        print(f'Viz skipped: {e}')
except Exception as e:
    print(f'Alpha Miner failed: {e}')
    net_alpha = im_alpha = fm_alpha = None


Alpha Miner:
  Places     : 180
  Transitions: 62
  Arcs       : 783
Saved -> figs/petri_net_alpha.png


### 5c. Alpha+ Miner

Extension of Alpha that correctly handles **length-one loops** (self-loops) and **length-two loops** (A->B->A), which appear frequently in iterative screening steps.

In [9]:
# -- 5c. Alpha+ Miner -------------------------------------------------------
try:
    net_alpha_plus, im_alpha_plus, fm_alpha_plus = pm4py.discover_petri_net_alpha_plus(log_discovery)
    print('Alpha+ Miner:')
    print(f'  Places     : {len(net_alpha_plus.places)}')
    print(f'  Transitions: {len(net_alpha_plus.transitions)}')
    print(f'  Arcs       : {len(net_alpha_plus.arcs)}')
    try:
        from pm4py.visualization.petri_net import visualizer as pn_vis
        gviz = pn_vis.apply(net_alpha_plus, im_alpha_plus, fm_alpha_plus, parameters={'format': 'png'})
        pn_vis.save(gviz, 'figs/petri_net_alpha_plus.png')
        print('Saved -> figs/petri_net_alpha_plus.png')
    except Exception as e:
        print(f'Viz skipped: {e}')
except Exception as e:
    print(f'Alpha+ Miner failed: {e}')
    net_alpha_plus = im_alpha_plus = fm_alpha_plus = None


C:\Users\feder\AppData\Local\Temp\ipykernel_30220\276371109.py:3: DeprecatedWarning: discover_petri_net_alpha_plus is deprecated as of 2.3.0 and will be removed in 3.0.0. This method will be removed in a future release.
  net_alpha_plus, im_alpha_plus, fm_alpha_plus = pm4py.discover_petri_net_alpha_plus(log_discovery)


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\feder\anaconda3\envs\pmthesis\Lib\site-packages\IPython\core\interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\feder\AppData\Local\Temp\ipykernel_30220\276371109.py", line 3, in <module>
    net_alpha_plus, im_alpha_plus, fm_alpha_plus = pm4py.discover_petri_net_alpha_plus(log_discovery)
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\feder\anaconda3\envs\pmthesis\Lib\site-packages\pm4py\util\deprecation.py", line 281, in _inner
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\feder\anaconda3\envs\pmthesis\Lib\site-packages\pm4py\discovery.py", line 483, in discover_petri_net_alpha_plus
    return alpha_miner.apply(
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\feder\anaconda3\envs\pmthesis\Lib\site-packages\pm4py\algo\discovery\alpha\algorithm.py", line 133

### 5d. Heuristics Miner

Uses **dependency scores** (frequency-weighted) to filter noisy edges before building the net. `dependency_threshold=0.5` keeps edges with score >= 0.5; raise it for a simpler (less-fitting) model. Generally the best trade-off for noisy real-life logs.

In [10]:
# -- 5d. Heuristics Miner (dependency_threshold=0.5) -----------------------
try:
    net_heu, im_heu, fm_heu = pm4py.discover_petri_net_heuristics(
        log_discovery,
        dependency_threshold=0.5,
        and_threshold=0.65,
        loop_two_threshold=0.5
    )
    print('Heuristics Miner (dep=0.5, and=0.65, loop2=0.5):')
    print(f'  Places     : {len(net_heu.places)}')
    print(f'  Transitions: {len(net_heu.transitions)}')
    print(f'  Arcs       : {len(net_heu.arcs)}')
    try:
        from pm4py.visualization.petri_net import visualizer as pn_vis
        gviz = pn_vis.apply(net_heu, im_heu, fm_heu, parameters={'format': 'png'})
        pn_vis.save(gviz, 'figs/petri_net_heuristics.png')
        print('Saved -> figs/petri_net_heuristics.png')
    except Exception as e:
        print(f'Viz skipped: {e}')
    try:
        hnet = pm4py.discover_heuristics_net(log_discovery, dependency_threshold=0.5)
        from pm4py.visualization.heuristics_net import visualizer as hn_vis
        gviz_hn = hn_vis.apply(hnet, parameters={'format': 'png'})
        hn_vis.save(gviz_hn, 'figs/heuristics_net.png')
        print('Heuristics dependency graph saved -> figs/heuristics_net.png')
    except Exception as e:
        print(f'Heuristics net viz skipped: {e}')
except Exception as e:
    print(f'Heuristics Miner failed: {e}')
    net_heu = im_heu = fm_heu = None


Heuristics Miner (dep=0.5, and=0.65, loop2=0.5):
  Places     : 246
  Transitions: 422
  Arcs       : 1140


dot: failure to create cairo surface: out of memory


Viz skipped: Command '[WindowsPath('dot'), '-Kdot', '-Tpng', '-O', 'tmpeltr8xxa.gv']' returned non-zero exit status 3221225477. [stderr: b'dot: failure to create cairo surface: out of memory\r\n']
Heuristics dependency graph saved -> figs/heuristics_net.png


### 5e. ILP Miner

Solves an **integer linear program** over the causal footprint matrix to find the minimal Petri net that fits the log exactly. Produces the most precise net but is computationally expensive -- can take several minutes on 20 K cases.

In [11]:
# -- 5e. ILP Miner (slow -- may take several minutes) ----------------------
import time
print('Running ILP Miner on 20K-case sample (may take a few minutes)...')
t0 = time.time()
try:
    net_ilp, im_ilp, fm_ilp = pm4py.discover_petri_net_ilp(log_discovery)
    elapsed = time.time() - t0
    print(f'ILP Miner (elapsed: {elapsed:.1f}s):')
    print(f'  Places     : {len(net_ilp.places)}')
    print(f'  Transitions: {len(net_ilp.transitions)}')
    print(f'  Arcs       : {len(net_ilp.arcs)}')
    try:
        from pm4py.visualization.petri_net import visualizer as pn_vis
        gviz = pn_vis.apply(net_ilp, im_ilp, fm_ilp, parameters={'format': 'png'})
        pn_vis.save(gviz, 'figs/petri_net_ilp.png')
        print('Saved -> figs/petri_net_ilp.png')
    except Exception as e:
        print(f'Viz skipped: {e}')
except Exception as e:
    print(f'ILP Miner failed: {e}')
    net_ilp = im_ilp = fm_ilp = None


Running ILP Miner on 20K-case sample (may take a few minutes)...


discovering Petri net using ILP miner, completed causal relations ::   0%|          | 0/360 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 5f. Process Tree (Inductive) -> Petri Net

The Inductive Miner first builds a **process tree** (hierarchical, block-structured model) which is then converted to a Petri net. The tree is sound by construction and useful for hierarchical process analysis. Both representations are saved.

In [ ]:
# -- 5f. Process Tree (Inductive) -> Petri net ------------------------------
try:
    process_tree = pm4py.discover_process_tree_inductive(log_discovery, noise_threshold=0.2)
    print(f'Process Tree (noise=0.2):')
    print(f'  {str(process_tree)[:120]}')
    try:
        from pm4py.visualization.process_tree import visualizer as pt_vis
        gviz_pt = pt_vis.apply(process_tree, parameters={'format': 'png'})
        pt_vis.save(gviz_pt, 'figs/process_tree_inductive.png')
        print('Process tree saved -> figs/process_tree_inductive.png')
    except Exception as e:
        print(f'Process tree viz skipped: {e}')
    from pm4py.convert import convert_to_petri_net
    net_pt, im_pt, fm_pt = convert_to_petri_net(process_tree)
    print('Converted Petri net:')
    print(f'  Places     : {len(net_pt.places)}')
    print(f'  Transitions: {len(net_pt.transitions)}')
    print(f'  Arcs       : {len(net_pt.arcs)}')
    try:
        from pm4py.visualization.petri_net import visualizer as pn_vis
        gviz = pn_vis.apply(net_pt, im_pt, fm_pt, parameters={'format': 'png'})
        pn_vis.save(gviz, 'figs/petri_net_process_tree.png')
        print('Saved -> figs/petri_net_process_tree.png')
    except Exception as e:
        print(f'Viz skipped: {e}')
except Exception as e:
    print(f'Process Tree failed: {e}')
    net_pt = im_pt = fm_pt = None


### 5g. Structural Comparison

Compare all discovered models by structural complexity (places, transitions, arcs). Fewer places/transitions = simpler model, but usually lower fitness. Token-replay fitness is added in Section 7.

In [ ]:
# -- 5g. Structural comparison of all discovered models --------------------
discovered_models = {}
candidates = [
    ('Inductive Miner IMf (noise=0.2)',  net_disc,       im_disc,       fm_disc),
    ('Alpha Miner',                       net_alpha,      im_alpha,      fm_alpha),
    ('Alpha+ Miner',                      net_alpha_plus, im_alpha_plus, fm_alpha_plus),
    ('Heuristics Miner (dep=0.5)',        net_heu,        im_heu,        fm_heu),
    ('ILP Miner',                         net_ilp,        im_ilp,        fm_ilp),
    ('Process Tree -> Petri net',         net_pt,         im_pt,         fm_pt),
    ('Normative (hand-crafted)',          net_norm,       im_norm,       fm_norm_positive),
]
for name, net, im, fm in candidates:
    if net is not None:
        discovered_models[name] = (net, im, fm)

print(f"{'Algorithm':<42} {'Places':>7} {'Trans':>7} {'Arcs':>6}")
print('-' * 65)
for name, (net, im, fm) in discovered_models.items():
    print(f'{name:<42} {len(net.places):>7} {len(net.transitions):>7} {len(net.arcs):>6}')


## 6. Normative Petri Net (hand-crafted)

In [ ]:
from pm4py.objects.petri_net.obj import PetriNet, Marking
from pm4py.objects.petri_net.utils import petri_utils

net_norm = PetriNet('normative_recruiting')

place_names = [
    'p_start', 'p_after_screen', 'p_scheduled', 'p_interview_decision',
    'p_interviewed', 'p_assessed', 'p_compensation', 'p_offer_review',
    'p_offer_decision', 'p_bgcheck', 'p_end_positive', 'p_end_negative'
]
places = {name: PetriNet.Place(name) for name in place_names}
for p in places.values():
    net_norm.places.add(p)

def add_t(net, name, label=None):
    t = PetriNet.Transition(name, label)
    net.transitions.add(t)
    return t

t_rec_screen  = add_t(net_norm, 't_rec_screen',  'Recruiter Screen')
t_mgr_screen  = add_t(net_norm, 't_mgr_screen',  'Manager Screen')
t_skip_mgr    = add_t(net_norm, 't_skip_mgr',    None)   # tau
t_schedule    = add_t(net_norm, 't_schedule',    'Schedule Interview')
t_int_dec     = add_t(net_norm, 't_int_dec',     'Make Interview Decision')
t_interview   = add_t(net_norm, 't_interview',   'Interview')
t_reinterview = add_t(net_norm, 't_reinterview', 'Re-Interview')
t_assess      = add_t(net_norm, 't_assess',      'Assess Candidate')
t_skip_assess = add_t(net_norm, 't_skip_assess', None)   # tau
t_compensation= add_t(net_norm, 't_compensation','Propose Compensation')
t_offer_review= add_t(net_norm, 't_offer_review','Review Offer Letter')
t_offer_dec   = add_t(net_norm, 't_offer_dec',   'Make Offer Decision')
t_bgcheck     = add_t(net_norm, 't_bgcheck',     'Make Background Check Decision')
t_hire        = add_t(net_norm, 't_hire',        'Ready for Hire')
t_rej_screen  = add_t(net_norm, 't_rej_screen',  'Reject at Screen')
t_rej_int     = add_t(net_norm, 't_rej_int',     'Reject at Interview')
t_rej_offer   = add_t(net_norm, 't_rej_offer',   'Reject Offer')
t_withdraw    = add_t(net_norm, 't_withdraw',    'Withdraw')

p = places
# Start -> recruiter screen -> optional manager screen -> scheduled
petri_utils.add_arc_from_to(p['p_start'],          t_rec_screen,  net_norm)
petri_utils.add_arc_from_to(t_rec_screen,          p['p_after_screen'], net_norm)
petri_utils.add_arc_from_to(p['p_after_screen'],   t_mgr_screen,  net_norm)
petri_utils.add_arc_from_to(t_mgr_screen,          p['p_scheduled'], net_norm)
petri_utils.add_arc_from_to(p['p_after_screen'],   t_skip_mgr,    net_norm)
petri_utils.add_arc_from_to(t_skip_mgr,            p['p_scheduled'], net_norm)
petri_utils.add_arc_from_to(p['p_after_screen'],   t_rej_screen,  net_norm)
petri_utils.add_arc_from_to(t_rej_screen,          p['p_end_negative'], net_norm)

# Schedule -> interview decision
petri_utils.add_arc_from_to(p['p_scheduled'],      t_schedule,    net_norm)
petri_utils.add_arc_from_to(t_schedule,            p['p_interview_decision'], net_norm)
petri_utils.add_arc_from_to(p['p_interview_decision'], t_int_dec, net_norm)
petri_utils.add_arc_from_to(t_int_dec,             p['p_interviewed'], net_norm)
petri_utils.add_arc_from_to(p['p_interview_decision'], t_rej_int, net_norm)
petri_utils.add_arc_from_to(t_rej_int,             p['p_end_negative'], net_norm)

# Interview + re-interview loop
petri_utils.add_arc_from_to(p['p_interviewed'],    t_interview,   net_norm)
petri_utils.add_arc_from_to(t_interview,           p['p_assessed'], net_norm)
petri_utils.add_arc_from_to(p['p_interviewed'],    t_reinterview, net_norm)
petri_utils.add_arc_from_to(t_reinterview,         p['p_interviewed'], net_norm)

# Optional assess
petri_utils.add_arc_from_to(p['p_assessed'],       t_assess,      net_norm)
petri_utils.add_arc_from_to(t_assess,              p['p_compensation'], net_norm)
petri_utils.add_arc_from_to(p['p_assessed'],       t_skip_assess, net_norm)
petri_utils.add_arc_from_to(t_skip_assess,         p['p_compensation'], net_norm)

# Offer flow
petri_utils.add_arc_from_to(p['p_compensation'],   t_compensation,net_norm)
petri_utils.add_arc_from_to(t_compensation,        p['p_offer_review'], net_norm)
petri_utils.add_arc_from_to(p['p_offer_review'],   t_offer_review,net_norm)
petri_utils.add_arc_from_to(t_offer_review,        p['p_offer_decision'], net_norm)
petri_utils.add_arc_from_to(p['p_offer_decision'], t_offer_dec,   net_norm)
petri_utils.add_arc_from_to(t_offer_dec,           p['p_bgcheck'], net_norm)
petri_utils.add_arc_from_to(p['p_offer_decision'], t_rej_offer,   net_norm)
petri_utils.add_arc_from_to(t_rej_offer,           p['p_end_negative'], net_norm)

# Background check & withdraw
petri_utils.add_arc_from_to(p['p_bgcheck'],        t_bgcheck,     net_norm)
petri_utils.add_arc_from_to(t_bgcheck,             p['p_end_positive'], net_norm)
petri_utils.add_arc_from_to(p['p_bgcheck'],        t_withdraw,    net_norm)
petri_utils.add_arc_from_to(t_withdraw,            p['p_end_negative'], net_norm)

im_norm            = Marking({p['p_start']: 1})
fm_norm_positive   = Marking({p['p_end_positive']: 1})
fm_norm_negative   = Marking({p['p_end_negative']: 1})

print(f'Normative net: {len(net_norm.places)} places, {len(net_norm.transitions)} transitions, {len(net_norm.arcs)} arcs')

try:
    from pm4py.visualization.petri_net import visualizer as pn_vis
    gviz = pn_vis.apply(net_norm, im_norm, fm_norm_positive, parameters={'format': 'png'})
    pn_vis.save(gviz, 'figs/petri_net_normative.png')
    print('Normative Petri net saved.')
except Exception as e:
    print(f'Petri net viz skipped: {e}')


## 7. Conformance -- Token Replay

Token replay is run on **all** models in `discovered_models` (built in Section 5g) using the 50 K-case conformance sample.

| Metric | Meaning |
|--------|---------|
| `fit`  | Average trace fitness (0-1): proportion of tokens correctly produced/consumed |
| `fit%` | Percentage of traces with perfect fitness |
| `prec` | ETC precision: how much of the model's allowed behaviour is actually observed |
| `simp` | Simplicity: inverse of arc density (higher = simpler) |

In [ ]:
# -- 7. Token Replay on all discovered models (50K-case sample) -----------
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness
from pm4py.algo.evaluation.precision import algorithm as precision_eval
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_eval

def token_replay_metrics(log, net, im, fm, label):
    replayed = token_replay.apply(log, net, im, fm)
    fitness  = replay_fitness.evaluate(replayed, variant=replay_fitness.Variants.TOKEN_BASED)
    avg_fit  = fitness.get('average_trace_fitness', float('nan'))
    pct_fit  = fitness.get('perc_fit_traces', float('nan'))
    try:
        prec = precision_eval.apply(log, net, im, fm)
    except Exception:
        prec = float('nan')
    simp = simplicity_eval.apply(net)
    print(f'  {label:<42}  fit={avg_fit:.4f}  fit%={pct_fit:.1f}%  prec={prec:.4f}  simp={simp:.4f}')
    return {'fitness': avg_fit, 'pct_fit': pct_fit, 'precision': prec,
            'simplicity': simp, 'replayed': replayed}

print(f'Token replay on {len(log_conformance)} traces (50K-case sample)')
print(f"  {'Algorithm':<42}  {'fit':>6}  {'fit%':>6}  {'prec':>6}  {'simp':>6}")
print('-' * 75)

tr_results = {}
for name, (net, im, fm) in discovered_models.items():
    try:
        tr_results[name] = token_replay_metrics(log_conformance, net, im, fm, name)
    except Exception as e:
        print(f'  {name:<42}  ERROR: {e}')

# Backward-compat aliases used by the Alignments cell
replayed_disc = tr_results.get('Inductive Miner IMf (noise=0.2)', {}).get('replayed')
replayed_norm = tr_results.get('Normative (hand-crafted)', {}).get('replayed')


## 8. Conformance — Alignments (2K cases, slow ~10–30 min)

In [ ]:
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments

print('Running alignments on 2K-case sample...')
aligned_disc = None
aligned_norm = None

try:
    aligned_disc = alignments.apply(log_alignment, net_disc, im_disc, fm_disc)
    fit_disc = replay_fitness.evaluate(aligned_disc,
                                        variant=replay_fitness.Variants.ALIGNMENT_BASED)
    print(f'Discovered model alignment fitness: {fit_disc}')
except Exception as e:
    print(f'Discovered alignment failed: {e}')

try:
    aligned_norm = alignments.apply(log_alignment, net_norm, im_norm, fm_norm_positive)
    fit_norm = replay_fitness.evaluate(aligned_norm,
                                        variant=replay_fitness.Variants.ALIGNMENT_BASED)
    print(f'Normative model alignment fitness: {fit_norm}')
except Exception as e:
    print(f'Normative alignment failed: {e}')


## 9. Deviation Analysis (normative model)

In [ ]:
if aligned_norm is not None:
    from collections import Counter
    move_on_log   = []
    move_on_model = []
    for trace_align in aligned_norm:
        for step in trace_align.get('alignment', []):
            if isinstance(step, tuple) and len(step) == 2:
                log_move, model_move = step
                if log_move != '>>' and model_move == '>>':
                    move_on_log.append(str(log_move))
                elif log_move == '>>' and model_move != '>>':
                    move_on_model.append(str(model_move))

    print('Top-10 Move-on-Log (in data, not in normative model):')
    for act, cnt in Counter(move_on_log).most_common(10):
        print(f'  {act}: {cnt}')

    print('\nTop-10 Move-on-Model (skipped normative transitions):')
    for act, cnt in Counter(move_on_model).most_common(10):
        print(f'  {act}: {cnt}')
else:
    print('Alignment results not available.')


## 10. Social Network Analysis — Resource Handoffs

In [ ]:
# Build resource-to-resource handoff graph
# For consecutive events in the same case with different resources, add edge resource_i -> resource_j

try:
    import networkx as nx
    
    print('Building resource handoff graph...')
    # Sample 20K cases for performance
    np.random.seed(42)
    sample_cases = np.random.choice(df['Case_id'].unique(), 
                                     size=min(20000, df['Case_id'].nunique()), 
                                     replace=False)
    df_sna = df[df['Case_id'].isin(sample_cases)].copy()
    
    # Build handoff edges
    from collections import defaultdict
    handoff_counts = defaultdict(int)
    handoff_counts_hired = defaultdict(int)
    handoff_counts_nothired = defaultdict(int)
    
    for case_id, grp in df_sna.groupby('Case_id'):
        grp = grp.sort_values('timestamp')
        resources = grp['Completed By'].fillna('SYSTEM').tolist()
        hired_flag = int(grp['hired'].iloc[0]) if pd.notna(grp['hired'].iloc[0]) else 0
        for i in range(len(resources) - 1):
            r_from, r_to = resources[i], resources[i+1]
            if r_from != r_to:
                handoff_counts[(r_from, r_to)] += 1
                if hired_flag == 1:
                    handoff_counts_hired[(r_from, r_to)] += 1
                else:
                    handoff_counts_nothired[(r_from, r_to)] += 1
    
    print(f'Total unique handoff pairs: {len(handoff_counts):,}')
    
    # Build graph with top-100 edges by frequency
    top_edges = sorted(handoff_counts.items(), key=lambda x: -x[1])[:100]
    G = nx.DiGraph()
    for (r_from, r_to), weight in top_edges:
        G.add_edge(r_from, r_to, weight=weight)
    
    # Compute centrality measures
    deg_centrality = nx.degree_centrality(G)
    try:
        bet_centrality = nx.betweenness_centrality(G, weight='weight')
    except Exception:
        bet_centrality = {}
    
    top_deg = sorted(deg_centrality.items(), key=lambda x: -x[1])[:10]
    top_bet = sorted(bet_centrality.items(), key=lambda x: -x[1])[:10]
    
    print('\nTop-10 Resources by Degree Centrality:')
    for r, c in top_deg:
        print(f'  {str(r)[:40]:40s}  {c:.4f}')
    
    print('\nTop-10 Resources by Betweenness Centrality:')
    for r, c in top_bet:
        print(f'  {str(r)[:40]:40s}  {c:.4f}')
    
    # Analyze handoffs in hired vs not-hired cases
    total_hired_cases = df_sna[df_sna['hired'].astype(float) == 1]['Case_id'].nunique()
    total_nothired_cases = df_sna[df_sna['hired'].astype(float) == 0]['Case_id'].nunique()
    
    total_handoffs_hired = sum(handoff_counts_hired.values())
    total_handoffs_nothired = sum(handoff_counts_nothired.values())
    
    avg_handoffs_hired = total_handoffs_hired / max(total_hired_cases, 1)
    avg_handoffs_nothired = total_handoffs_nothired / max(total_nothired_cases, 1)
    
    print(f'\nAvg handoffs per hired case    : {avg_handoffs_hired:.2f}')
    print(f'Avg handoffs per not-hired case: {avg_handoffs_nothired:.2f}')
    
    # Visualize: spring layout of top-50 nodes by degree
    top50_nodes = sorted(G.degree(), key=lambda x: -x[1])[:50]
    top50_node_set = {n for n, d in top50_nodes}
    G_sub = G.subgraph(top50_node_set).copy()
    
    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(G_sub, seed=42, k=2)
    node_sizes = [300 * deg_centrality.get(n, 0.01) * 20 + 50 for n in G_sub.nodes()]
    edge_weights = [G_sub[u][v]['weight'] for u, v in G_sub.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths = [1 + 3 * w / max_w for w in edge_weights]
    
    nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes, 
                            node_color='steelblue', alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G_sub, pos, width=edge_widths, 
                            edge_color='gray', alpha=0.5, 
                            connectionstyle='arc3,rad=0.1', ax=ax,
                            arrows=True, arrowsize=10)
    # Label only top-10 by degree
    top10_labels = {n: str(n)[:15] for n, d in top50_nodes[:10]}
    nx.draw_networkx_labels(G_sub, pos, labels=top10_labels, font_size=7, ax=ax)
    ax.set_title('Resource Handoff Network (top-50 nodes, top-100 edges)')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('figs/social_network.png', bbox_inches='tight')
    plt.close()
    print('Social network saved to figs/social_network.png')
    
except ImportError:
    print('networkx not installed. Run: pip install networkx')
except Exception as e:
    print(f'SNA failed: {e}')
    import traceback; traceback.print_exc()
